# Step 22 — Merge individual and monthly pick indexes

Run after steps 20 and 21. This notebook contains the merge implementation directly; only shared ID normalization/time helpers and configuration remain imported. It reads the two pick-index CSVs and replaces the merged CSV and, when configured, the suppressed-pick report beneath the external output root. Legacy source files and input indexes are unchanged.

“Primary” means the workflow prefers individual-file picks to monthly picks when both describe a matching arrival; it does not establish that those picks are scientifically correct. Event IDs remain source grouping keys. This step neither creates physical earthquakes nor associates waveforms/hypocenters. No clock correction is applied.

## 1. Review configuration

Set `CONFIG_FILE` if needed and inspect input/output paths and tolerances. This setup cell does not execute the workflow.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    primary=P['INDIV_PICK_INDEX'],
    secondary=P['MONTHLY_PICK_INDEX'],
    out=P['MERGED_PICK_INDEX'],
    time_tolerance=float(P['PICK_TIME_TOL']),
    report=P['MERGED_PICKS_SUPPRESSED'],
)
display(SETTINGS)

## 2. Imports and merge helpers

The normalization helper uses the existing SEED ID where available. Parsed UTC-style times are a representation of source labels, not independent timing validation.

Within each source, rows are sorted by event ID, normalized SEED ID, phase and time. Duplicate suppression compares each pick to the immediately preceding sorted pick within the group, using an inclusive tolerance. Chains of nearby picks may therefore be suppressed relative to preceding rows that were themselves suppressed; this is the preserved algorithm, not a new clustering rule.

Monthly picks are then compared to the nearest primary pick with the same normalized SEED ID and phase, within the configured inclusive tolerance (normally 0.5 s). Cross-source suppression does not require matching event IDs and is not a one-to-one event association. Surviving monthly picks retain their original block IDs.

Invalid times are dropped and counted in diagnostics, but are not included in the suppressed-pick CSV. The `primary_in`/`secondary_in` counters describe time-valid rows, while separate counters record invalid times. Retained picks receive `pick_priority`; suppressed rows receive a reason. Raw provenance columns are carried through unchanged.

In [ ]:
from pathlib import Path

import pandas as pd

from pinatubo_steps.shared import (normalize_seed_id, to_dt)

def parse_time_col(df: pd.DataFrame, col: str) -> pd.Series:
    return to_dt(df[col])

def dedupe_within_event(
    df: pd.DataFrame,
    time_col: str,
    tol_s: float,
):
    """
    Deduplicate picks within each event_id.

    For each (event_id, seed_norm, phase),
    picks within tol_s are duplicates.
    Keeps first occurrence.
    """
    if df.empty:
        return df.copy(), df.iloc[0:0].copy()

    df = df.sort_values(
        ["event_id", "seed_norm", "phase", time_col]
    ).reset_index(drop=True)

    key_cols = ["event_id", "seed_norm", "phase"]
    df["_dt"] = df.groupby(key_cols)[time_col].diff().dt.total_seconds()

    dup_mask = df["_dt"].notna() & (df["_dt"].abs() <= tol_s)

    suppressed = df.loc[dup_mask].drop(columns="_dt")
    kept = df.loc[~dup_mask].drop(columns="_dt")

    return kept.reset_index(drop=True), suppressed.reset_index(drop=True)

def merge_pick_tables(
    df_primary: pd.DataFrame,
    df_secondary: pd.DataFrame,
    tolerance: float,
):

    # ------------------------------------------------------------------
    # Normalize & parse times
    # ------------------------------------------------------------------

    df_primary = df_primary.copy()
    df_secondary = df_secondary.copy()

    df_primary["seed_norm"] = df_primary.apply(normalize_seed_id, axis=1)
    df_secondary["seed_norm"] = df_secondary.apply(normalize_seed_id, axis=1)

    df_primary["pick_time_dt"] = parse_time_col(df_primary, "pick_time")
    df_secondary["pick_time_dt"] = parse_time_col(df_secondary, "pick_time")

    # Drop unparseable times
    prim_bad = df_primary["pick_time_dt"].isna()
    sec_bad = df_secondary["pick_time_dt"].isna()

    df_primary = df_primary.loc[~prim_bad].reset_index(drop=True)
    df_secondary = df_secondary.loc[~sec_bad].reset_index(drop=True)

    # ------------------------------------------------------------------
    # 1) Deduplicate within events
    # ------------------------------------------------------------------

    df_primary_kept, df_primary_dups = dedupe_within_event(
        df_primary,
        time_col="pick_time_dt",
        tol_s=tolerance,
    )

    df_secondary_kept, df_secondary_dups = dedupe_within_event(
        df_secondary,
        time_col="pick_time_dt",
        tol_s=tolerance,
    )

    # ------------------------------------------------------------------
    # 2) Suppress monthly picks duplicating authoritative picks
    # ------------------------------------------------------------------

    prim = df_primary_kept[
        ["seed_norm", "phase", "pick_time_dt"]
    ].sort_values(["seed_norm", "phase", "pick_time_dt"])

    sec = df_secondary_kept.sort_values(
        ["seed_norm", "phase", "pick_time_dt"]
    )

    keep_sec = []
    suppress_sec = []

    for (seed_norm, phase), sec_block in sec.groupby(
        ["seed_norm", "phase"], sort=False
    ):
        prim_block = prim[
            (prim["seed_norm"] == seed_norm) &
            (prim["phase"] == phase)
        ]

        if prim_block.empty:
            keep_sec.append(sec_block)
            continue

        m = pd.merge_asof(
            sec_block,
            prim_block,
            on="pick_time_dt",
            direction="nearest",
            tolerance=pd.Timedelta(seconds=tolerance),
            suffixes=("", "_prim"),
        )

        #matched = m["pick_time_dt_prim"].notna()
        matched = m["seed_norm_prim"].notna()
        suppress_sec.append(sec_block.loc[matched.values])
        keep_sec.append(sec_block.loc[~matched.values])

    df_secondary_final = (
        pd.concat(keep_sec, ignore_index=True)
        if keep_sec else sec.iloc[0:0].copy()
    )

    df_secondary_suppressed = (
        pd.concat(suppress_sec, ignore_index=True)
        if suppress_sec else sec.iloc[0:0].copy()
    )

    # ------------------------------------------------------------------
    # 3) Build merged output
    # ------------------------------------------------------------------

    df_primary_kept["pick_priority"] = "primary"
    df_secondary_final["pick_priority"] = "secondary"

    merged = pd.concat(
        [df_primary_kept, df_secondary_final],
        ignore_index=True,
    )

    merged = merged.drop(
        columns=["seed_norm", "pick_time_dt"],
        errors="ignore",
    )

    suppressed = pd.concat(
        [
            df_primary_dups.assign(
                suppressed_reason="primary_duplicate_within_event"
            ),
            df_secondary_dups.assign(
                suppressed_reason="secondary_duplicate_within_event"
            ),
            df_secondary_suppressed.assign(
                suppressed_reason="secondary_duplicate_of_primary"
            ),
        ],
        ignore_index=True,
    ).drop(
        columns=["seed_norm", "pick_time_dt"],
        errors="ignore",
    )

    diagnostics = {
        "primary_in": int(len(df_primary)),
        "primary_bad_time": int(prim_bad.sum()),
        "primary_dups_removed": int(len(df_primary_dups)),
        "primary_out": int(len(df_primary_kept)),
        "secondary_in": int(len(df_secondary)),
        "secondary_bad_time": int(sec_bad.sum()),
        "secondary_dups_removed": int(len(df_secondary_dups)),
        "secondary_suppressed_by_primary": int(len(df_secondary_suppressed)),
        "secondary_out": int(len(df_secondary_final)),
        "merged_out": int(len(merged)),
    }

    return merged, suppressed, diagnostics

def run(
    *,
    primary,
    secondary,
    out,
    time_tolerance=0.5,
    report=None
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        primary=primary,
        secondary=secondary,
        out=out,
        time_tolerance=time_tolerance,
        report=report,
    )

    df_primary = pd.read_csv(args.primary, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})
    df_secondary = pd.read_csv(args.secondary, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})

    merged, suppressed, diag = merge_pick_tables(
        df_primary, df_secondary, args.time_tolerance
    )

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out, index=False)

    print("\n=== STEP 22 SUMMARY ===")
    for k, v in diag.items():
        print(f"{k:35s}: {v}")
    print(f"Output CSV: {out}")

    if args.report:
        rpt = Path(args.report)
        rpt.parent.mkdir(parents=True, exist_ok=True)
        suppressed.to_csv(rpt, index=False)
        print(f"Suppressed CSV: {rpt}")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## Run and review

The next cell executes the workflow and writes configured products. Inspect the summary and outputs before continuing.

In [ ]:
result = run(**SETTINGS)

In [3]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)


df_primary


,raw_line,raw_lineno,event_id,event_id_source,pick_id,pha_file,pha_path,station,channel,seed_id,phase,pick_time,onset,first_motion,weight
0,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_1,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,P,1991-06-03T07:39:38.490000Z,NaN,U,0
1,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_562,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,P,1991-06-03T07:39:38.490000Z,NaN,U,0
2,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_2,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,S,1991-06-03T07:39:39.990000Z,NaN,NaN,2
3,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_563,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,S,1991-06-03T07:39:39.990000Z,NaN,NaN,2
4,BURZ P 4 910603073938.05,7,91060300,individual_pha_filename,indpha_91060300_3,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUR,EHZ,XB.BUR..EHZ,P,1991-06-03T07:39:38.050000Z,NaN,NaN,4


df_secondary


,raw_line,raw_lineno,event_id,event_id_source,pick_id,pha_file,station,channel,seed_id,phase,pick_time,onset,first_motion,weight
0,BUGZEPU1 91 73116 616.67,3,pinaug91_block0000,monthly_phase_block,monpha_pinaug91_block0000_1,pinaug91.pha,BUG,EHZ,XB.BUG..EHZ,P,1991-07-31T16:06:16.670000Z,E,U,1.0
1,CABZEP 1 91 73116 615.76,4,pinaug91_block0000,monthly_phase_block,monpha_pinaug91_block0000_2,pinaug91.pha,CAB,EHZ,XB.CAB..EHZ,P,1991-07-31T16:06:15.760000Z,E,NaN,1.0
2,CRWZ PU0 91 73116 619.51,2,pinaug91_block0000,monthly_phase_block,monpha_pinaug91_block0000_3,pinaug91.pha,CRW,EHZ,XB.CRW..EHZ,P,1991-07-31T16:06:19.510000Z,NaN,U,0.0
3,PI2ZIPD0 91 73116 614.81,1,pinaug91_block0000,monthly_phase_block,monpha_pinaug91_block0000_4,pinaug91.pha,PI2,EHZ,XB.PI2..EHZ,P,1991-07-31T16:06:14.810000Z,I,D,0.0
4,BUGZIPD0 91 73116 718.88,10,pinaug91_block0001,monthly_phase_block,monpha_pinaug91_block0001_5,pinaug91.pha,BUG,EHZ,XB.BUG..EHZ,P,1991-07-31T16:07:18.880000Z,I,D,0.0


merged


,raw_line,raw_lineno,event_id,event_id_source,pick_id,pha_file,pha_path,station,channel,seed_id,phase,pick_time,onset,first_motion,weight,pick_priority
0,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_1,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,P,1991-06-03T07:39:38.490000Z,NaN,U,0.0,primary
1,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_2,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,S,1991-06-03T07:39:39.990000Z,NaN,NaN,2.0,primary
2,BURZ P 4 910603073938.05,7,91060300,individual_pha_filename,indpha_91060300_3,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUR,EHZ,XB.BUR..EHZ,P,1991-06-03T07:39:38.050000Z,NaN,NaN,4.0,primary
3,CABZ P 5 910603073953.23,9,91060300,individual_pha_filename,indpha_91060300_4,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,CAB,EHZ,XB.CAB..EHZ,P,1991-06-03T07:39:53.230000Z,NaN,NaN,5.0,primary
4,GRNZ P 5 910603073953.23,2,91060300,individual_pha_filename,indpha_91060300_5,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,GRN,EHZ,XB.GRN..EHZ,P,1991-06-03T07:39:53.230000Z,NaN,NaN,5.0,primary


suppressed


,raw_line,raw_lineno,event_id,event_id_source,pick_id,pha_file,pha_path,station,channel,seed_id,phase,pick_time,onset,first_motion,weight,suppressed_reason
0,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_562,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,P,1991-06-03T07:39:38.490000Z,NaN,U,0.0,primary_duplicate_within_event
1,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_563,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,S,1991-06-03T07:39:39.990000Z,NaN,NaN,2.0,primary_duplicate_within_event
2,BURZ P 4 910603073938.05,7,91060300,individual_pha_filename,indpha_91060300_564,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUR,EHZ,XB.BUR..EHZ,P,1991-06-03T07:39:38.050000Z,NaN,NaN,4.0,primary_duplicate_within_event
3,CABZ P 5 910603073953.23,9,91060300,individual_pha_filename,indpha_91060300_565,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,CAB,EHZ,XB.CAB..EHZ,P,1991-06-03T07:39:53.230000Z,NaN,NaN,5.0,primary_duplicate_within_event
4,GRNZ P 5 910603073953.23,2,91060300,individual_pha_filename,indpha_91060300_566,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,GRN,EHZ,XB.GRN..EHZ,P,1991-06-03T07:39:53.230000Z,NaN,NaN,5.0,primary_duplicate_within_event


diag


{'primary_in': 1330,
 'primary_bad_time': 0,
 'primary_dups_removed': 211,
 'primary_out': 1119,
 'secondary_in': 57042,
 'secondary_bad_time': 0,
 'secondary_dups_removed': 134,
 'secondary_suppressed_by_primary': 247,
 'secondary_out': 56661,
 'merged_out': 57780}

## Continue

Check `result["diag"]` and the suppressed-pick report. Retained secondary groups may overlap primary groups without being merged into the same event ID. Continue to step 23 for diagnostics and the later association steps for waveform/hypocenter matching.